# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

### Obtención del corpus
Primero descargamos nltk para obtener el corpus de Project Gutneberg

In [5]:
%pip install nltk pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import nltk
nltk.download('gutenberg')
nltk.download('stopwords')

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [19]:
import re
import pandas as pd
from nltk.corpus import gutenberg,stopwords
stopwords_ingles = set(stopwords.words("english"))
def preprocesar(texto):
    #primero normalizamos el texto
    texto = texto.lower()
    #extraemos las secuencias de letras
    terminos = re.findall(r"[a-z]+",texto)
    #eliminamos las stopwords
    terminos = [termino for termino in terminos if termino not in stopwords_ingles]
    return terminos
# guardamos los documentos originales
documentos = {}
for identificador in gutenberg.fileids():
    texto_original = gutenberg.raw(identificador)
    terminos_procesados = preprocesar(texto_original)
    documentos[identificador] = {
        "texto_original": texto_original,
        "terminos_procesados": terminos_procesados
    }
print("Documentos preprocesados", len(documentos))


Documentos preprocesados 18


### Construimos la tabla solicitada

In [24]:
filas = []
for identificador, datos in documentos.items():
    texto = datos["texto_original"]
    encabezado = re.search(r"\[([^\]]+)\]",texto[:500])
    if encabezado:
        titulo = encabezado.group(1)
    else:
        titulo = "Título no encontrado"
    filas.append(
        {
            "identificador de documento": identificador,
            "titulo": titulo,
            "Número de términos únicos": len(set(datos["terminos_procesados"])),
        }
    )
tabla_documentos = pd.DataFrame(filas)
display(tabla_documentos)

,identificador de documento,titulo,Número de términos únicos
0,austen-emma.txt,Emma by Jane Austen 1816,6965
1,austen-persuasion.txt,Persuasion by Jane Austen 1818,5607
2,austen-sense.txt,Sense and Sensibility by Jane Austen 1811,6148
3,bible-kjv.txt,The King James Bible,12443
4,blake-poems.txt,Poems by William Blake 1789,1400
5,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryan...,3696
6,burgess-busterbrown.txt,The Adventures of Buster Bear by Thornton W. B...,1382
7,carroll-alice.txt,Alice's Adventures in Wonderland by Lewis Carr...,2423
8,chesterton-ball.txt,The Ball and The Cross by G.K. Chesterton 1909,8029
9,chesterton-brown.txt,The Wisdom of Father Brown by G. K. Chesterton...,7589


### Mostramos características del corpus

In [26]:
numero_documentos = len(documentos)
longitudes = []
vocabulario = set()
for identificador,datos in documentos.items():
    terminos = datos["terminos_procesados"]
    longitud = len(terminos)
    longitudes.append(longitud)
    vocabulario.update(terminos)
longitud_minima = min(longitudes)
longitud_media= sum(longitudes)/len(longitudes)
longitud_maxima = max(longitudes)
print("Número de documentos:", numero_documentos)
print("tamaño del vocabulario:", len(vocabulario))
print("longitud mínima:", longitud_minima)
print("longitud media:", longitud_media)
print("longitud maxima:", longitud_maxima)



Número de documentos: 18
tamaño del vocabulario: 41357
longitud mínima: 3805
longitud media: 56987.5
longitud maxima: 374945


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [38]:
from collections import Counter
import math
def calcular_tf(terminos):
    return Counter(terminos)
tf_documentos= {}
for identificador, datos in documentos.items():
    terminos = datos["terminos_procesados"]
    tf_documentos[identificador] = calcular_tf(terminos)
def calcular_df(documentos):
    df = Counter()
    for datos in documentos.values():
        terminos = datos["terminos_procesados"]
        terminos_unicos = set(terminos)
        df.update(terminos_unicos)
    return df
df_corpus = calcular_df(documentos)
def calcular_idf(df_corpus, numero_documentos):
    idf = {}
    for termino, frecuencia_documentos in df_corpus.items():
        idf[termino] = math.log(numero_documentos / frecuencia_documentos) +1
    return idf
numero_documentos = len(documentos)
idf_corpus = calcular_idf(df_corpus, numero_documentos)

def calcular_tfidf(tf_documentos, idf_corpus):
    tfidf_documentos = {}

    for identificador, frecuencias in tf_documentos.items():
        pesos = {}

        for termino, tf in frecuencias.items():
            pesos[termino] = tf * idf_corpus[termino]

        tfidf_documentos[identificador] = pesos

    return tfidf_documentos
tfidf_documentos = calcular_tfidf(tf_documentos, idf_corpus)

#imprimimos ejemplos
identificador = "austen-emma.txt"
termino = "emma"

print("TF:", tf_documentos[identificador][termino])
print("DF:", df_corpus[termino])
print("IDF:", idf_corpus.get(termino))
print("TF-IDF:", tfidf_documentos[identificador].get(termino, 0))


TF: 865
DF: 2
IDF: 3.1972245773362196
TF-IDF: 2765.59925939583


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

### Vocabulario ordenado

In [42]:
# utilizamos sorted para ordenar
vocabulario_ordenado = sorted(vocabulario)
#imprimimos un ejemplo
print("Cantidad de documentos:", len(ids_documentos))
print("Cantidad de términos:", len(vocabulario_ordenado))
print("Primeros 10 términos:", vocabulario_ordenado[:10])


Cantidad de documentos: 18
Cantidad de términos: 41357
Primeros 10 términos: ['aaron', 'aaronites', 'ab', 'aback', 'abaddon', 'abaft', 'abagtha', 'abana', 'abandon', 'abandoned']


### Contar las frecuencias TF

In [43]:
ids_documentos = list(documentos.keys())
tf_documentos = {}
for identificador in ids_documentos:
    terminos = documentos[identificador]["terminos_procesados"]
    tf_documentos[identificador] = calcular_tf(terminos)
# imprimimos un ejemplo
print(tf_documentos["austen-emma.txt"]["emma"])

865


### Calcular DF contando cada documento una sola vez por término.

In [46]:
df_corpus = calcular_df(documentos)
# imprimimos un ejemplo
print("DF de 'emma':", df_corpus["emma"])

DF de 'emma': 2


### Calcular IDF con la función de la sección anterior.

In [47]:
numero_documentos = len(ids_documentos)
idf_corpus = calcular_idf(df_corpus, numero_documentos)
#imprimimos un ejemplo
print("IDF de 'emma':", idf_corpus.get("emma", "No encontrado"))

IDF de 'emma': 3.1972245773362196


### Obtener la matriz documento-término TF-IDF.

In [50]:
from scipy.sparse import csr_matrix
filas = []
columnas = []
valores = []
indice_terminos = {}
for indice, termino in enumerate(vocabulario_ordenado):
    indice_terminos[termino] = indice

for indice_documento, identificador in enumerate(ids_documentos):
    # Counter con los TF del documento actual
    frecuencias = tf_documentos[identificador]
    for termino, tf in frecuencias.items():
        indice_termino = indice_terminos[termino]
        # Peso TF-IDF = TF × IDF
        peso = tf * idf_corpus[termino]
        filas.append(indice_documento)
        columnas.append(indice_termino)
        valores.append(peso)
matriz_tfidf = csr_matrix(
    (valores, (filas, columnas)),
    shape=(len(ids_documentos), len(vocabulario_ordenado))
)
print("Dimensiones de la matriz:", matriz_tfidf.shape)

Dimensiones de la matriz: (18, 41357)


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [55]:
from sklearn.feature_extraction.text import TfidfVectorizer
corpus_procesado = []
for identificador in ids_documentos:
    terminos = documentos[identificador]["terminos_procesados"]
    corpus_procesado.append(terminos)
def devolver_terminos(terminos):
    return terminos

vectorizador = TfidfVectorizer(
    analyzer=devolver_terminos,
    vocabulary=indice_terminos,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None
)
matriz_sklearn = vectorizador.fit_transform(corpus_procesado)
print("Dimensiones de tu matriz:", matriz_tfidf.shape)
print("Dimensiones de scikit-learn:", matriz_sklearn.shape)
diferencia = matriz_tfidf - matriz_sklearn
if diferencia.nnz == 0:
    diferencia_maxima = 0.0
else:
    diferencia_maxima = abs(diferencia.data).max()

print("Máxima diferencia absoluta:", diferencia_maxima)

Dimensiones de tu matriz: (18, 41357)
Dimensiones de scikit-learn: (18, 41357)
Máxima diferencia absoluta: 0.0


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.